In [1]:
import torch
import copy
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from medmnist import PneumoniaMNIST, INFO
from sklearn.metrics import roc_auc_score
import wandb
from pathlib import Path

if "__file__" in globals():
    SCRIPT_DIR = Path(__file__).resolve().parent
else:
    SCRIPT_DIR = Path.cwd()
# print(f"当前脚本路径: {SCRIPT_DIR}")

In [2]:
config = {
    "learning_rate": 0.001,
    "batch_size": 64,
    "epochs": 35,
    "optimizer": "Adam",
    "weight_decay": 5e-5,
    "label_smoothing": 0.1,
    "seed": 42,
    "model": "CNN_3x3_BN_PneumoniaMNIST",
}
wandb.init(
    project="pneumonia-mnist-cnn",
    name="baseline-local2",
    config=config
)
torch.manual_seed(config["seed"])

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\35314\_netrc.
wandb: Currently logged in as: 3531427435 (3531427435-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: setting up run xu392t89
wandb: Tracking run with wandb version 0.30.0
wandb: Run data is saved locally in E:\code\PycharmProjects\pythonProject\pytorch\pneumonia_mnist\wandb\run-20260916_151612-xu392t89
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run baseline-local2
wandb:  View project at https://wandb.ai/3531427435-none/pneumonia-mnist-cnn
wandb:  View run at https://wandb.ai/3531427435-none/pneumonia-mnist-cnn/runs/xu392t89


In [3]:
train_transform = transforms.Compose([
# 【想一想，暂时不用改】28x28 补 4 像素黑边再裁回 28x28，相当于给训练集加平移抖动。
# 但在 28x28 这么小的图上 4 像素占比不小，而且黑边经过 Normalize 会变成约 -3.4 的强负值，
# 验证/测试集又没有这个过程。可以先保留把流程跑通，跑通后再关掉它做一次对照，用 AUC 说话。
#     transforms.RandomCrop(28, padding=4),
    transforms.ToTensor(),
# 【要改】这里的 (0.5719) 括号只起分组作用，它在 Python 里就是数字 0.5719，不是元组。
# 改成 [0.5719] 或 (0.5719,)（注意那个逗号）。原因：Normalize 是按通道给均值和标准差，
# 应该传序列；现在能跑通只是 torchvision 当成标量做了广播，属于碰巧对，
# 而且和下面 test_transform 的写法不一致。
    transforms.Normalize((0.5719,), (0.1684,)),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5719,), (0.1684,)),
])


# 【要改】MedMNIST 的标签是形状 (1,) 的数组，不是标量——你打印出来的 [1] 就是这个。
# 直接送进 CrossEntropyLoss 会报错：0D or 1D target tensor expected, multi-target not supported。
# 更危险的是只在 loss 那一行 squeeze 的半修版：下面 evaluate() 里的 (predicted == label)
# 会广播成 (batch, batch)，准确率能被算成 100% 以上而且不报错（实测 4 个样本会算出 8）。
# 建议在这里一次性解决：给 PneumoniaMNIST 传 target_transform，让取出来的标签就是标量。
# 这样 loss、准确率、后面的 AUC 全都自然正确。下面三行都要加上。
target_transform = lambda x: x[0]
train_set = PneumoniaMNIST(root=SCRIPT_DIR / 'data',split="train", transform=train_transform, target_transform=target_transform, download=True)
val_set = PneumoniaMNIST(root=SCRIPT_DIR / 'data',split="val", transform=test_transform, target_transform=target_transform, download=True)
test_set = PneumoniaMNIST(root=SCRIPT_DIR / 'data',split="test", transform=test_transform, target_transform=target_transform, download=True)

train_loader = DataLoader(train_set, batch_size=config["batch_size"], shuffle=True)
val_loader = DataLoader(val_set, batch_size=config["batch_size"], shuffle=False)
test_loader = DataLoader(test_set, batch_size=config["batch_size"], shuffle=False)

print(f"训练集 {len(train_set)} 张，验证集 {len(val_set)} 张，测试集 {len(test_set)} 张")
image, label = train_set[0]
print(f"图像尺寸: {image.shape}, 标签: {label}")
d = np.load(SCRIPT_DIR / 'data' / 'pneumoniamnist.npz')
print(d.files)
print(np.bincount(d['train_labels'].ravel()))

训练集 4708 张，验证集 524 张，测试集 624 张
图像尺寸: torch.Size([1, 28, 28]), 标签: 1
['train_images', 'val_images', 'test_images', 'train_labels', 'val_labels', 'test_labels']
[1214 3494]


In [4]:
# 计算mean和std
# C = 1
# sum_ = torch.zeros(C)
# sum_sq = torch.zeros(C)
# n = 0
# for images, _ in train_loader:
#     b = images.size(0)
#     images = images.view(b, C, -1)
#     sum_ += images.sum(dim=(0, 2))
#     sum_sq += (images ** 2).sum(dim=(0, 2))
#     n += b * images.size(2)
#
# mean = sum_ / n
# std = (sum_sq / n - mean ** 2) ** 0.5

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 2)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.classifier(x)
        return x

net = Net().to(device)
print("参数量:", sum(p.numel() for p in net.parameters()))
wandb.watch(net, log="all", log_freq=500)

Using device: cuda
参数量: 1673570


In [6]:
# 【不用改，但要写进文档】训练集两类是 1214 : 3494（约 1 : 2.9），基线阶段故意不做加权或重采样。
# 文档里要明确写：已知类别不平衡、基线不做处理、留作后续对照，否则会被当成遗漏。
criterion = nn.CrossEntropyLoss(label_smoothing=config["label_smoothing"])
optimizer = optim.Adam(
net.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"]
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, config["epochs"])

In [7]:
# 【要改】这个函数现在只返回准确率。本任务的官方指标是 AUC，而且两类不平衡
# （什么都不学、全猜肺炎就有 74.2% 的准确率），只看准确率会骗人。要做两件事：
#   1) 在循环里收集肺炎的分数，用 sklearn.metrics.roc_auc_score 算 AUC；
#   2) 让函数同时返回 (准确率, AUC)。
# 关键点：AUC 要的是连续分数，不是 argmax 之后的类别。outputs 是两列 logits，
# 可以直接取第 1 列，也可以先 softmax 再取第 1 列——想一想这两种做法算出的 AUC 是否一样，为什么。
def evaluate(model, loader):
    """在验证集或测试集上评估，返回 (准确率, 肺炎类 AUC)。"""
    model.eval()
    correct = 0
    total = 0
    all_labels = []    # 整个数据集所有样本的真实标签
    all_scores = []    # 整个数据集所有样本的"肺炎"分数

    with torch.no_grad():
        for image, label in loader:
            image, label = image.to(device), label.to(device)
            outputs = model(image)
            # 取"肺炎"那一列的连续分数：outputs 是 (batch, 2)，dim=1 是类别那一维
            pneumonia_score = torch.softmax(outputs, dim=1)[:, 1]
            _, predicted = torch.max(outputs, 1)
            total += label.size(0)
            correct += (predicted == label).sum().item()
             # 先收集，等整个数据集跑完再统一算 AUC
            all_labels.append(label.cpu())
            all_scores.append(pneumonia_score.cpu())
    all_labels = torch.cat(all_labels).numpy()
    all_scores = torch.cat(all_scores).numpy()
    accuracy = 100 * correct / total
    auc = roc_auc_score(all_labels, all_scores)
    return accuracy, auc

num_epochs = config["epochs"]
best_val_auc = 0.0
best_state = None

for epoch in range(num_epochs):
    net.train()
    running_loss = 0.0
# 【建议】改成 images, labels：这里的 image/label 会覆盖前面 train_set[0] 取出的那两个变量，
# 不影响结果，但读代码时容易混。
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = net(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    val_acc, val_auc = evaluate(net, val_loader)
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]
# 【要改】用准确率挑最佳权重，在不平衡数据上会偏向多数类。
# 改成用验证集 AUC 来挑，例如 if val_auc > best_val_auc:，相关变量名一起改。
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state = copy.deepcopy(net.state_dict())
    print(
        f"Epoch [{epoch+1}/{num_epochs}], "
        f"Loss: {avg_loss:.4f}, "
        f"Val AUC: {val_auc:.4f}%, "
        f"Best Val: {best_val_auc:.4f}%, "
        f"LR: {current_lr:.6f}"
    )
    wandb.log({
        "epoch": epoch + 1,
        "train/loss": avg_loss,
        "val/accuracy": val_acc,
        "val/auc": val_auc,
        "learning_rate": current_lr,
    })
print("Finished Training")

net.load_state_dict(best_state)
test_acc, test_auc = evaluate(net, test_loader)
print(f"验证集最佳 AUC: {best_val_auc:.4f}")
print(f"测试集 AUC: {test_auc:.4f}")
print(f"测试集准确率: {test_acc:.2f}%")

wandb.log({
    "best_val/auc": best_val_auc,
    "test/auc": test_auc,
    "test/accuracy": test_acc,
})

# 【要改】相对路径又会出问题：notebook 里会存到 pneumonia_mnist 目录，
# 换成 python train.py 从别的目录启动就换地方，服务器上更不可控。
# 用 SCRIPT_DIR / "outputs" / "pneumonia_model.pth"，并在脚本版里先把 outputs 目录建好
# （你的 fashion_mnist/train.py 就是这么处理的）。
torch.save(best_state, str(SCRIPT_DIR / "outputs" / "pneumonia_model.pth"))
artifact = wandb.Artifact(
    name="pneumonia_model",
    type="model"
)
# 【要改】同上，用绝对路径：
# artifact.add_file(str(SCRIPT_DIR / "outputs" / "pneumonia_model.pth"))
artifact.add_file(str(SCRIPT_DIR / "outputs" / "pneumonia_model.pth"))
wandb.log_artifact(artifact)
wandb.finish()

Epoch [1/35], Loss: 0.3319, Val AUC: 0.9918%, Best Val: 0.9918%, LR: 0.000998
Epoch [2/35], Loss: 0.2737, Val AUC: 0.9958%, Best Val: 0.9958%, LR: 0.000992
Epoch [3/35], Loss: 0.2629, Val AUC: 0.9964%, Best Val: 0.9964%, LR: 0.000982
Epoch [4/35], Loss: 0.2553, Val AUC: 0.9965%, Best Val: 0.9965%, LR: 0.000968
Epoch [5/35], Loss: 0.2482, Val AUC: 0.9958%, Best Val: 0.9965%, LR: 0.000950
Epoch [6/35], Loss: 0.2413, Val AUC: 0.9967%, Best Val: 0.9967%, LR: 0.000929
Epoch [7/35], Loss: 0.2381, Val AUC: 0.9968%, Best Val: 0.9968%, LR: 0.000905
Epoch [8/35], Loss: 0.2322, Val AUC: 0.9964%, Best Val: 0.9968%, LR: 0.000877
Epoch [9/35], Loss: 0.2264, Val AUC: 0.9962%, Best Val: 0.9968%, LR: 0.000846
Epoch [10/35], Loss: 0.2267, Val AUC: 0.9961%, Best Val: 0.9968%, LR: 0.000812
Epoch [11/35], Loss: 0.2202, Val AUC: 0.9961%, Best Val: 0.9968%, LR: 0.000775
Epoch [12/35], Loss: 0.2172, Val AUC: 0.9954%, Best Val: 0.9968%, LR: 0.000737
Epoch [13/35], Loss: 0.2171, Val AUC: 0.9959%, Best Val: 0.99

wandb: uploading artifact pneumonia_model; updating run metadata
wandb: uploading artifact pneumonia_model
wandb: uploading artifact pneumonia_model; uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 32-35, summary
wandb: 
wandb: Run history:
wandb:  best_val/auc ▁
wandb:         epoch ▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
wandb: learning_rate ██████▇▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▂▁▁▁▁▁▁▁
wandb: test/accuracy ▁
wandb:      test/auc ▁
wandb:    train/loss █▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:  val/accuracy ▁▄▆▆▅▇▅▇▇█▆█▅█▆▆█▇▇▇▆▅▇▇▆█▇▇▇▇▇▇▇▇▇
wandb:       val/auc ▁▇▇█▇██▇▇▇▇▆▇▇▇▇▆▆▄▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆
wandb: 
wandb: Run summary:
wandb:  best_val/auc 0.99682
wandb:         epoch 35
wandb: learning_rate 0
wandb: test/accuracy 88.30128
wandb:      test/auc 0.95193
wandb:    train/loss 0.20533
wandb:  val/accuracy 97.51908
wandb:       val/auc 0.99556
wandb: 
wandb:  View run baseline-local2 at: https://wandb.ai/3531427435-none/pneumonia-mnist-cnn/runs/xu392t89
wandb: 